# Generative Adversarial Networks (GANs)

## Colorização de Paisagens com Pix2Pix

Nos laboratórios anteriores, utilizamos *autoencoders* com arquitetura U-Net para colorizar imagens de paisagens em escala de cinza. O modelo aprendia a prever os canais de cor AB do espaço CIELAB a partir do canal de luminosidade L, minimizando o erro quadrático médio (MSE) pixel a pixel.

Embora funcional, essa abordagem tende a produzir cores desbotadas e dessaturadas, um fenômeno conhecido como "efeito sépia". Isso acontece porque o MSE penaliza igualmente todas as previsões e, diante da ambiguidade de cor (um céu pode ser azul, laranja ou roxo), o modelo aprende a prever a média das cores plausíveis, que é sempre uma cor neutra.

A proposta deste laboratório é explorar uma solução um pouco mais robusta, usando uma arquitetura denominada de Pix2Pix.

A ideia central de uma GAN é treinar dois modelos em competição, sendo:

- **Gerador (G)**: aprende a produzir imagens coloridas a partir da entrada em escala de cinza.
- **Discriminador (D)**: aprende a distinguir imagens coloridas reais das produzidas pelo modelo Gerador.

Essa competição força o modelo Gerador a produzir cores cada vez mais convincentes, superando a limitação do MSE puro.

<figure style="text-align: center;">
  <img src="assets/figura-1.jpg" alt="Figura 1" width="600">
</figure>

**Figura 1** – Arquitetura padrão de uma GAN, detalhando o fluxo de treinamento e a retropropagação entre o modelo do Gerador e o Discriminador.

## A Arquitetura Pix2Pix

O Pix2Pix [[Isola et al., 2017](https://arxiv.org/abs/1611.07004)] é uma GAN condicional (cGAN), na qual tanto o Gerador quanto o Discriminador recebem a imagem de entrada (em escala de cinza) como condição. Isso garante que a colorização gerada seja semanticamente consistente com a entrada.

O Discriminador PatchGAN não classifica a imagem inteira como real ou falsa, mas sim cada *patch* (região) de 70×70 pixels. Isso força o modelo Discriminador a avaliar a textura local, onde as diferenças de qualidade de cor mais aparecem.

<figure style="text-align: center;">
  <img src="assets/figura-2.png" alt="Figura 2" width="400">
</figure>

**Figura 2** – Arquitetura do modelo cGAN para colorização de imagens, composta por um gerador U-Net, responsável por predizer os canais de cromaticidade AB a partir do canal de luminância L, e um discriminador PatchGAN, responsável por distinguir entre pares reais e gerados.

## Preparação dos Dados

In [ ]:
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from skimage import color
from pathlib import Path
import tensorflow as tf
import numpy as np
import zipfile
import time
import os

from tqdm import tqdm

In [ ]:
# reprodutibilidade
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
# hiperparâmetros globais
IMG_SIZE = 128 # resolução das imagens (128×128)
BATCH_SIZE = 16 # batches menores são comuns em GANs por estabilidade
EPOCHS = 45 # GANs precisam de mais épocas que autoencoders

# constantes do espaço CIELAB (mesmo que o lab de autoencoders)
L_RANGE = 100.0 # canal L: de 0 a 100
AB_RANGE = 128.0 # canais AB: de -128 a +127

Será utilizado o mesmo conjunto de dados de paisagens e o mesmo *pipeline* de pré-processamento do laboratório de *autoencoders*, no espaço de cor CIELAB.

In [ ]:
# baixa o dataset landscapes do google drive (mesmo do lab3)
%pip install -q gdown

import gdown

FILE_ID     = "1luDUiWUP4RCNUOoBS1QtucIBc60LNmQt"
DATASET_ZIP = "assets/landscapes.zip"
DATASET_DIR = "dataset_landscapes"

gdown.download(id=FILE_ID, output=DATASET_ZIP)

In [ ]:
def setup_dataset(zip_path, extract_to):
    # extrai os dados compactados a partir de um arquivo .zip
    
    path = Path(extract_to)
    if not path.exists():
        print(f"Extraindo {zip_path}...")
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_to)
            print("Extração concluída.")
        else:
            print(f"{zip_path} não encontrado.")
    else:
        print(f"Diretório {extract_to} já existente.")
    return path

data_dir = setup_dataset(DATASET_ZIP, DATASET_DIR)

## Conversão para o espaço CIELAB

O espaço CIELAB separa a informação de brilho (L) da informação de cor (AB). Isso é ideal para colorização porque podemos usar L como entrada e pedir ao modelo que preveja AB como saída, tarefa exata o que o modelo do Gerador irá executar.

- **L** normalizado: `[0, 100] → [0, 1]`
- **AB** normalizados: `[-128, 127] → [-1, 1]`

**OBS**: Note que a saída do Gerador usará `tanh`.

In [ ]:
def rgb_to_lab_norm(image_rgb):
    # converte uma imagem RGB para CIELAB e normaliza os canais

    lab_image = color.rgb2lab(image_rgb)

    # luminosidade: 0 a 100
    l_channel  = lab_image[:, :, 0]
    # cromaticidade: -128 a +127
    ab_channels = lab_image[:, :, 1:]

    # normaliza para ranges que o modelo consegue aprender melhor
    l_norm  = l_channel / L_RANGE # [0, 1]
    ab_norm = ab_channels / AB_RANGE # [-1, 1] aproximado

    h, w = l_norm.shape
    l_norm = l_norm.reshape((h, w, 1))

    return l_norm.astype(np.float32), ab_norm.astype(np.float32)

In [ ]:
def tf_process_lab(image_path):
    # lê um arquivo de imagem do disco e converte para o par (L_norm, AB_norm)

    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    l_norm, ab_norm = tf.numpy_function(
        func=rgb_to_lab_norm,
        inp=[img],
        Tout=[tf.float32, tf.float32]
    )

    l_norm.set_shape([IMG_SIZE, IMG_SIZE, 1])
    ab_norm.set_shape([IMG_SIZE, IMG_SIZE, 2])

    return l_norm, ab_norm

In [ ]:
# lista todos os arquivos .jpg do dataset e divide em treino/val/teste
list_ds = tf.data.Dataset.list_files(
    str(DATASET_DIR + "/*.jpg"),
    shuffle=True,
    seed=SEED
)

val_size   = int(len(list_ds) * 0.15)
test_size  = int(len(list_ds) * 0.15)
train_size = int(len(list_ds)) - val_size - test_size

train_ds_lab = list_ds.take(train_size)
val_ds_lab   = list_ds.skip(train_size).take(val_size)
test_ds_lab  = list_ds.skip(train_size + val_size)

print(f"Treino:    {train_size} imagens")
print(f"Validação: {val_size} imagens")
print(f"Teste:     {test_size} imagens")

In [ ]:
def configure_for_performance(ds):
    # mapeia a conversão CIELAB, agrupa em batches e pré-carrega na memória

    ds = ds.map(tf_process_lab, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

In [ ]:
train_lab = configure_for_performance(train_ds_lab)
val_lab   = configure_for_performance(val_ds_lab)
test_lab  = configure_for_performance(test_ds_lab)

# verificação dos shapes
for l_batch, ab_batch in train_lab.take(1):
    print(f"L  (entrada):  {l_batch.shape}") # (16, 128, 128, 1)
    print(f"AB (alvo):     {ab_batch.shape}") # (16, 128, 128, 2)

## Definindo o Gerador Usando U-Net com Skip Connections

O Gerador do Pix2Pix é uma arquitetura U-Net, já explorada anteriormente, sendo composta por uma rede *encoder-decoder* com *skip connections* que conectam camadas simétricas do codificador ao decodificador.

Em tarefas de tradução de imagem (como colorização), a entrada e a saída compartilham muito da mesma estrutura espacial. As *skip connections* permitem que informações de baixo nível (e.g., bordas, texturas) sejam copiadas diretamente do codificador para o decodificador, sem precisar ser reconstituídas a partir do gargalo. Isso é especialmente importante para preservar os detalhes das bordas da paisagem.

<figure style="text-align: center;">
  <img src="assets/figura-3.png" alt="Figura 3" width="600">
</figure>

**Figura 3** – Arquitetura do modelo Gerador U-Net utilizado para colorização de imagens, com representação das dimensões espaciais e do número de canais em cada etapa do *encoder*, *bottleneck* e *decoder*, incluindo as conexões de salto (*skip connections*) entre camadas de mesma resolução.

No autoencoder definido em laboratórios passados, foi utilizado `UpSampling2D + Conv2D` para evitar artefatos de xadrez. No Pix2Pix, é implementado `Conv2DTranspose` com `BatchNormalization` e `Dropout` no decodificador, seguindo o design original proposto no artigo. O `BatchNorm` é essencial para estabilizar o treinamento adversarial.

In [ ]:
def downsample_block(filters, kernel_size=4, apply_batchnorm=True):
    # bloco de downsampling, Conv2D com stride 2 para reduzir a resolução pela metade
    # apply_batchnorm=False apenas para a primeira camada (sem normalização na entrada)

    block = models.Sequential()
    block.add(layers.Conv2D(
        filters, kernel_size,
        strides=2,
        padding="same",
        # bias é desnecessário quando BatchNorm está ativo
        use_bias=False
    ))
    if apply_batchnorm:
        block.add(layers.BatchNormalization())
    block.add(layers.LeakyReLU(0.2))  # LeakyReLU evita "dying ReLU" no discriminador
    return block

In [ ]:
def upsample_block(filters, kernel_size=4, apply_dropout=False):
    # bloco de upsampling, Conv2DTranspose com stride 2 para dobrar a resolução
    # apply_dropout=True nas primeiras camadas do decodificador (regularização extra)

    block = models.Sequential()
    block.add(layers.Conv2DTranspose(
        filters, kernel_size,
        strides=2,
        padding="same",
        use_bias=False
    ))
    block.add(layers.BatchNormalization())
    if apply_dropout:
        # dropout de 50% nas 3 primeiras camadas do decoder
        block.add(layers.Dropout(0.5))
    block.add(layers.ReLU())
    return block

In [ ]:
def build_generator():
    # gerador u-net, recebe L (1 canal) e produz AB (2 canais)

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

    # definição do codificador
    # cada bloco reduz a resolução pela metade e dobra os filtros
    enc1 = downsample_block(64,  apply_batchnorm=False)(inputs) # 64×64
    enc2 = downsample_block(128)(enc1) # 32×32
    enc3 = downsample_block(256)(enc2) # 16×16
    enc4 = downsample_block(512)(enc3) # 8×8
    enc5 = downsample_block(512)(enc4) # 4×4

    # bottleneck)
    bottleneck = downsample_block(512)(enc5) # 2×2

    # definição do decodificador
    # as skip connections concatenam a saída do encoder com a do decoder
    # Dropout nas 3 primeiras camadas do decoder = mais robustez ao ruído
    dec1 = upsample_block(512, apply_dropout=True)(bottleneck)
    dec1 = layers.Concatenate()([dec1, enc5]) # skip enc5

    dec2 = upsample_block(512, apply_dropout=True)(dec1)
    dec2 = layers.Concatenate()([dec2, enc4]) # skip enc4

    dec3 = upsample_block(256, apply_dropout=True)(dec2)
    dec3 = layers.Concatenate()([dec3, enc3]) # skip enc3

    dec4 = upsample_block(128)(dec3)
    dec4 = layers.Concatenate()([dec4, enc2]) # skip enc2

    dec5 = upsample_block(64)(dec4)
    dec5 = layers.Concatenate()([dec5, enc1]) # skip enc1

    # definição da camada de saída
    # tanh produz saídas em [-1, 1], compatível com a normalização de AB
    outputs = layers.Conv2DTranspose(
        2, # 2 canais de saída: canal a e canal b
        kernel_size=4,
        strides=2,
        padding="same",
        activation="tanh"
    )(dec5) # saída: 128×128×2

    return models.Model(inputs, outputs, name="Gerador_Pix2Pix")

In [ ]:
generator = build_generator()
generator.summary()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  O modelo Gerador utiliza skip connections entre o codificador e o decodificador. Por que essa escolha é especialmente relevante para a tarefa de colorização? O que seria perdido se o modelo fosse um autoencoder simples sem essas conexões?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Definindo o Discriminador com PatchGAN

O Discriminador do Pix2Pix é um PatchGAN, ao invés de produzir um único escalar, indicando "real/falso" para a imagem inteira, ele produz uma grade de decisões, onde cada posição avalia um patch de ~70×70 pixels da imagem.

- **Avalia textura local**: cores realistas se manifestam em detalhes locais (e.g., textura de grama, gradiente de céu). O PatchGAN penaliza erros justamente   nessa escala.
- **Mais eficiente**: parâmetros muito menores que um discriminador global.
- **Mais estável**: o sinal de gradiente é mais rico e local, facilitando o treinamento.

O Discriminador recebe dois canais concatenados como entrada: o canal `L` de entrada (condição) e os canais `AB` (reais ou gerados). Isso é o que torna o Pix2Pix uma GAN condicional (cGAN), o discriminador aprende não apenas "isso parece real?", mas "isso parece uma colorização real *para esta entrada específica*?"

<figure style="text-align: center;">
  <img src="assets/figura-4.png" alt="Figura 4" width="300">
</figure>

**Figura 4** – Arquitetura conceitual do discriminador PatchGAN, que recebe a concatenação dos canais de luminância L e cromaticidade AB e produz uma grade de classificações locais, avaliando a realismo da colorização em diferentes regiões da imagem.

In [ ]:
def build_discriminator():
    # discriminador patchgan, classifica patches de ~70×70 pixels como real/falso
    # recebe a concatenação de (L_input, AB_colorizado) como entrada condicional

    # entradas separadas para poder acessá-las individualmente se necessário
    input_l  = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="input_L")
    input_ab = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 2), name="input_AB")

    # concatena condição (L) com a imagem a ser avaliada (AB real ou fake)
    x = layers.Concatenate()([input_l, input_ab])    # (128, 128, 3)

    # blocos de downsampling do patchgan
    x = downsample_block(64,  apply_batchnorm=False)(x) # 64×64
    x = downsample_block(128)(x) # 32×32
    x = downsample_block(256)(x) # 16×16

    # zero padding + conv sem stride para ajuste de receptive field
    x = layers.ZeroPadding2D()(x) # 18×18
    x = layers.Conv2D(512, 4, use_bias=False)(x) # 15×15
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # camada de saída: um valor por patch (sem ativação = logits para BCEWithLogits)
    x = layers.ZeroPadding2D()(x) # 17×17
    patch_output = layers.Conv2D(1, 4)(x) # 14×14×1 (grade de patches)

    return models.Model([input_l, input_ab], patch_output, name="Discriminador_PatchGAN")

In [ ]:
discriminator = build_discriminator()
discriminator.summary()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
  O PatchGAN classifica regiões locais de ~70×70 pixels em vez da imagem inteira. Que vantagem isso traz para avaliar a qualidade de uma colorização? Pense em qual escala os erros de cor costumam aparecer.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Funções de Perda

O Pix2Pix usa uma combinação de duas funções de perda:

O Discriminador é treinado para:
- Dar **saída alta** (real) quando recebe `(L, AB_real)`
- Dar **saída baixa** (falso) quando recebe `(L, AB_fake)`

Desse modo, usar *binary cross-entropy com logits* é uma solução mais estável numericamente. Sua formulação é definido por:

$$
\mathcal{L}_D =
-\mathbb{E}_{(L, AB_{\mathrm{real}})}
\left[
\log D(L, AB_{\mathrm{real}})
\right]
-
\mathbb{E}_{L}
\left[
\log\left(1 - D(L, G(L))\right)
\right]
$$

O Gerador quer enganar o modelo Discriminador (fazer a saída ser classificada como real), mas também quer ser fiel à imagem original (perda L1). O coeficiente `LAMBDA` controla o quanto a fidelidade pixel a pixel importa, sendo definido por:

$$
\mathcal{L}_G =
-\mathbb{E}_{L}
\left[
\log D(L, G(L))
\right]
+
\lambda
\left\|
AB_{\mathrm{real}} - G(L)
\right\|_1
$$

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 3:</b><br>
  A loss do modelo Gerador combina uma componente adversarial (BCE) com uma componente de fidelidade (L1), ponderada por LAMBDA=100. O que aconteceria se LAMBDA fosse zero? E se fosse muito alto (ex.: 1000)? Em qual dos dois extremos o resultado se pareceria mais com o autocodificador do feito no laboratório anterior?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
# binary cross-entropy aplicada aos logits (saída do discriminador sem sigmoide)
bce_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# coeficiente de peso da perda L1 na loss total do gerador
LAMBDA = 100

def discriminator_loss(real_output, fake_output):
    # perda do discriminador: maximizar acerto em reais e falsos
    # real_output: saída do discriminador para (L, AB_real), queremos 1
    # fake_output: saída do discriminador para (L, AB_fake), queremos 0
    
    loss_real = bce_loss(tf.ones_like(real_output) * 0.9, real_output)
    loss_fake = bce_loss(tf.zeros_like(fake_output), fake_output)
    return loss_real + loss_fake

def generator_loss(fake_output, ab_generated, ab_real):
    # perda do gerador: enganar o discriminador + fidelidade pixel a pixel (L1)
    # perda adversarial: queremos que o discriminador classifique AB_fake como real (1)

    adv_loss = bce_loss(tf.ones_like(fake_output), fake_output)

    # perda L1: fidelidade pixel a pixel entre AB gerado e AB real
    l1_loss = tf.reduce_mean(tf.abs(ab_real - ab_generated))

    return adv_loss + LAMBDA * l1_loss

## Otimizadores

O Pix2Pix usa Adam com *learning rate* de $2.10^{-4}$ e `beta_1=0.5` (ao invés do padrão 0.9). O `beta_1` menor faz o otimizador "esquecer" o momento mais rapidamente, isso é importante em GANs porque a *loss* muda de natureza a cada passo (ora treinamos o gerador, ora o discriminador).

In [ ]:
# otimizadores separados para gerador e discriminador
# beta_1=0.5 é o padrão do paper pix2pix (mais estável que 0.9 para GANs)
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# métricas para acompanhar durante o treino
gen_loss_metric  = tf.keras.metrics.Mean(name="gen_loss")
disc_loss_metric = tf.keras.metrics.Mean(name="disc_loss")

## Treinamento do Modelo

Em GANs, o treinamento não usa o `model.fit()` do Keras. Em vez disso, implementamos um loop manual com `@tf.function` para compilar o passo de treinamento em um grafo TensorFlow, o que acelera significativamente a execução.

```
Para cada batch (L, AB_real):

  1. G(L) -> AB_fake [Gerador produz colorização]
  
  2. D(L, AB_real)-> real_output [Discriminador avalia o real]
  
  3. D(L, AB_fake) -> fake_output [Discriminador avalia o falso]
  
  4. Calcula disc_loss -> atualiza D [D aprende a distinguir]
  
  5. Calcula gen_loss -> atualiza G [G aprende a enganar D + fidelidade L1]
```

Note que o Discriminador e o Gerador são atualizados juntos em cada passo, mas com gradientes separados.

In [ ]:
@tf.function
def train_step(l_batch, ab_real):
    # executa um passo de treinamento adversarial para um batch

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        # gerador produz AB colorizado a partir de L
        ab_fake = generator(l_batch, training=True)

        # discriminador avalia par real (L, AB_real) e par falso (L, AB_fake)
        real_output = discriminator([l_batch, ab_real],  training=True)
        fake_output = discriminator([l_batch, ab_fake],  training=True)

        # calcula as losses
        gen_loss  = generator_loss(fake_output, ab_fake, ab_real)
        disc_loss = discriminator_loss(real_output, fake_output)

    # calcula e aplica os gradientes — separados para cada rede
    gen_grads  = gen_tape.gradient(gen_loss,  generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(
        zip(gen_grads,  generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(disc_grads, discriminator.trainable_variables)
    )

    # acumula métricas para exibição
    gen_loss_metric.update_state(gen_loss)
    disc_loss_metric.update_state(disc_loss)

Para monitorar o treinamento de GANs, é fundamental visualizar amostras geradas periodicamente. Ao contrário do *autoencoder*, não temos uma métrica de validação direta, o que nos guia é a qualidade visual das imagens geradas.

In [ ]:
def lab_to_rgb_denorm(l_norm, ab_norm):
    # reconstrói a imagem RGB a partir dos canais CIELAB normalizados

    l_channel   = l_norm * L_RANGE    # [0,1] → [0,100]
    ab_channels = ab_norm * AB_RANGE  # [-1,1] → [-128,128]
    lab_image   = np.dstack((l_channel, ab_channels))
    return color.lab2rgb(lab_image)

def show_gan_progress(l_batch, ab_real_batch, epoch, num=3):
    # gera e exibe amostras do gerador para monitoramento visual do treino

    ab_fake_batch = generator(l_batch[:num], training=False).numpy()

    plt.figure(figsize=(12, num * 3))

    for i in range(num):
        l      = l_batch[i].numpy()
        ab_r   = ab_real_batch[i].numpy()
        ab_f   = ab_fake_batch[i]

        img_original = lab_to_rgb_denorm(l[:, :, 0], ab_r)
        img_gerada   = lab_to_rgb_denorm(l[:, :, 0], ab_f)

        plt.subplot(num, 3, i * 3 + 1)
        plt.imshow(l[:, :, 0], cmap="gray")
        plt.title("Entrada (L)")
        plt.axis("off")

        plt.subplot(num, 3, i * 3 + 2)
        plt.imshow(img_gerada)
        plt.title(f"Pix2Pix (época {epoch})")
        plt.axis("off")

        plt.subplot(num, 3, i * 3 + 3)
        plt.imshow(img_original)
        plt.title("Original")
        plt.axis("off")

    plt.suptitle(f"Época {epoch}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

# batch fixo de validação para comparar o progresso ao longo das épocas
sample_l, sample_ab = next(iter(val_lab))

O loop percorre as épocas, executa `train_step` em cada batch e exibe amostras visuais a cada 5 épocas para acompanhar o progresso qualitativo.

In [ ]:
# %pip install --upgrade jupyter ipywidgets

In [ ]:
def get_lr(epoch):
    # mantém o lr constante na primeira metade e decai linearmente a 0 na
    # segunda metade
    decay_start = EPOCHS // 2
    if epoch <= decay_start:
        return 2e-4
    else:
        # fração de quanto já andamos na segunda metade
        progress = (epoch - decay_start) / (EPOCHS - decay_start)
        return 2e-4 * (1.0 - progress)

In [ ]:
# comente essa célula e descomente a próxima se quiser pular a etapa de treinamento

CHECKPOINT_DIR = Path("saved_models/pix2pix")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

best_gen_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    new_lr = get_lr(epoch)
    generator_optimizer.learning_rate.assign(new_lr)
    discriminator_optimizer.learning_rate.assign(new_lr)

    gen_loss_metric.reset_state()
    disc_loss_metric.reset_state()

    # barra de progresso por batch dentro da época
    pbar = tqdm(
        train_lab,
        desc=f"Época {epoch:3d}/{EPOCHS}",
        unit="batch",
        leave=False
    )

    for l_batch, ab_real in pbar:
        train_step(l_batch, ab_real)

        # atualiza os valores ao vivo na barra a cada batch
        pbar.set_postfix({
            "G": f"{gen_loss_metric.result():.4f}",
            "D": f"{disc_loss_metric.result():.4f}"
        })

    gen_loss_val  = gen_loss_metric.result().numpy()
    disc_loss_val = disc_loss_metric.result().numpy()
    elapsed       = time.time() - start

    # resumo da época
    print(
        f"[{epoch:3d}/{EPOCHS}]  "
        f"Gen Loss: {gen_loss_val:.4f}  |  "
        f"Disc Loss: {disc_loss_val:.4f}  |  "
        f"{elapsed:.1f}s"
    )

    if gen_loss_val < best_gen_loss:
        best_gen_loss = gen_loss_val
        generator.save(str(CHECKPOINT_DIR / "best_generator.keras"))
        print(f"melhor gerador salvo (gen_loss={gen_loss_val:.4f})")
    # salva snapshot a cada 5 épocas para recuperar o melhor visual depois
    if epoch % 5 == 0:
        generator.save(str(CHECKPOINT_DIR / f"generator_epoch_{epoch:03d}.keras"))

    if epoch % 5 == 0 or epoch == 1:
        show_gan_progress(sample_l, sample_ab, epoch)

In [ ]:
# descomente essa célula para carregar os pesos já treinados

# # baixe os pesos do modelo pix2pix já treinados
# %pip install -q gdown

# import gdown

# FILE_ID = "1UBJD2VMDaAwEtwN2nzrfXSIgEIW204_Z"
# DATASET = "assets/gans_pix2px.keras"

# gdown.download(
#     id=FILE_ID,
#     output=DATASET
# )

# # carrega o melhor gerador salvo durante o treinamento
# best_generator = tf.keras.models.load_model(DATASET)

## Avaliação do Modelo

O que buscamos observar nas imagens geradas podem ser explicitadas em três critérios: (a) coerência semântica (e.g., o céu ficou azul, a vegetação ficou verde?), (b) saturação (e.g., as cores têm vivacidade ou são desbotadas como no as obtidas em laboratórios anteriores?) e ausência de artefatos (e.g., manchas, bordas coloridas, pixels isolados). 

In [ ]:
def show_final_results(dataset, model, num=5):
    # exibe os resultados finais: entrada em P&B, colorização do pix2pix e original

    l_batch, ab_batch = next(iter(dataset))
    ab_preds = model(l_batch[:num], training=False).numpy()

    plt.figure(figsize=(12, num * 3))
    for i in range(num):
        l    = l_batch[i].numpy()
        ab_r = ab_batch[i].numpy()
        ab_f = ab_preds[i]

        img_original = lab_to_rgb_denorm(l[:, :, 0], ab_r)
        img_pix2pix  = lab_to_rgb_denorm(l[:, :, 0], ab_f)

        plt.subplot(num, 3, i * 3 + 1)
        plt.imshow(l[:, :, 0], cmap="gray")
        plt.title("Entrada (escala de cinza)")
        plt.axis("off")

        plt.subplot(num, 3, i * 3 + 2)
        plt.imshow(img_pix2pix)
        plt.title("Pix2Pix (GAN)")
        plt.axis("off")

        plt.subplot(num, 3, i * 3 + 3)
        plt.imshow(img_original)
        plt.title("Original")
        plt.axis("off")

    plt.suptitle("Resultados Finais — Pix2Pix", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_final_results(test_lab, best_generator)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 4:</b><br>
  Observando as imagens geradas pelo Pix2Pix no conjunto de teste, avalie os resultados usando os três critérios (a), (b) e (c) definidos na seção de avaliação.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Análise dos Resultados


| Aspecto | U-Net + MSE (autoencoders) | Pix2Pix |
|---|---|---|
| **Saturação** | Cores desbotadas ("sépia") | Cores vibrantes |
| **Coerência semântica** | Média | Discriminador aprende estrutura |
| **Diversidade de cores** | Média das opções plausíveis | Gera escolhas consistentes |
| **Artefatos** | Raramente | Possível (*mode collapse* é um risco) |
| **Custo de treino** | Baixo | Alto, dois modelos competindo |


O MSE penaliza o erro quadrático uniformemente em todos os pixels. Diante de ambiguidade (um pôr do sol pode ser laranja, rosa ou roxo), o gradiente empurra o modelo em direções opostas e ele converge para a média, gerando uma cor neutra.

O Discriminador do Pix2Pix resolve esse problema de outra forma, na qual ele avalia se as cores parecem reais comparadas às do conjunto de dados. Uma cor neutra "de compromisso" não passa no discriminador, então o Gerador é forçado a fazer escolhas.

Sobre algumas limitações observadas, podemos destacar:

- **Mode collapse**: o modelo Gerador pode aprender a produzir sempre a mesma paleta de cores independente da entrada. Se isso ocorrer, a perda L1 (com LAMBDA alto) ajuda a ancorar.
- **Instabilidade**: as losses de GANs não convergem monotonicamente como as de autoencoders. Uma `gen_loss` crescendo pode ser boa notícia (o discriminador está melhorando).

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 5:</b><br>
  Pesquise e explique o fenômeno de mode collapse em GANs. Por que ele é considerado um dos principais problemas de treinamento adversarial?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Referências

Isola, P., Zhu, J. Y., Zhou, T., & Efros, A. A. (2017). Image-to-image translation with   conditional adversarial networks. *CVPR 2017*. https://arxiv.org/abs/1611.07004